<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Embeding_Trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch transformers pandas tqdm numpy

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

from transformers import AutoTokenizer
from transformers import EsmModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

In [ ]:
protein_sequences = df["target_sequence"].tolist()
valid_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

def clean_sequence(seq):
    seq = seq.upper()

    seq = "".join(
        aa for aa in seq
        if aa in valid_amino_acids
    )

    return seq


protein_sequences = [
    clean_sequence(seq)
    for seq in protein_sequences
]


In [ ]:
MODEL_NAME = "facebook/esm2_t12_35M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = EsmModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

In [ ]:
import torch
import numpy as np
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

# Settings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

BATCH_SIZE = 128

loader = DataLoader(
    protein_sequences,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
    num_workers=4,
    persistent_workers=True
)

embeddings = []

# Extract Embeddings

with torch.inference_mode():

    for batch in tqdm(loader):

        tokens = tokenizer(
            list(batch),
            padding=True,
            truncation=True,
            max_length=1024,
            return_tensors="pt"
        )

        input_ids = tokens["input_ids"].to(device, non_blocking=True)
        attention_mask = tokens["attention_mask"].to(device, non_blocking=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Mean Pooling
        emb = (
            outputs.last_hidden_state *
            attention_mask.unsqueeze(-1)
        ).sum(dim=1)

        emb = emb / attention_mask.sum(dim=1, keepdim=True)

        embeddings.append(emb.cpu())

protein_embeddings = torch.cat(embeddings).numpy()

np.save("protein_embeddings.npy", protein_embeddings)

In [ ]:
batch_size = 16

embeddings = []

for i in range(0, len(protein_sequences), batch_size):

    batch = protein_sequences[i:i + batch_size]

    inputs = tokenizer(
        batch,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        emb = outputs.last_hidden_state.mean(dim=1)

    embeddings.append(
        emb.cpu().numpy()
    )

protein_embeddings = np.concatenate(
    embeddings,
    axis=0
)